In [ ]:
# Install LangGraph — the core library for building graph-based AI workflows.
!pip install -q langgraph

In [ ]:
import time  # Used to simulate slow network calls (e.g., fetching data from external APIs)

from IPython.display import Image
from google.colab import userdata
from langchain_core.runnables import Runnable, RunnableConfig
from langgraph.checkpoint.memory import InMemorySaver  # In-memory checkpoint backend
from langgraph.graph import END, START, StateGraph
from langgraph.graph.state import CompiledStateGraph
from pathlib import Path
from typing import Annotated, TypedDict

# Helper: renders the compiled graph as a PNG image and shows it inline in Jupyter
def display_graph(runnable: Runnable, output_png: Path) -> None:
    with output_png.open(mode="wb") as file:
        file.write(runnable.get_graph().draw_mermaid_png())

    display(Image(output_png, format="png"))

# Helper: prints the graph's state at each execution step.
# Useful to see how parallel nodes affect the state over time.
def explore_state_history(compiled_state_graph: CompiledStateGraph, config: RunnableConfig):
    state_history = list(compiled_state_graph.get_state_history(config))

    for snapshot in reversed(state_history):
        print(f"Step: {snapshot.metadata['step']}")
        print("Current state:")
        print(snapshot.values)
        print(f"Next: {snapshot.next}")
        print()

In [ ]:
# A checkpointer is needed to enable state history tracking
checkpointer = InMemorySaver()

# Custom merge function for the 'findings' field.
# When multiple nodes write to 'findings' simultaneously (during parallel execution),
# LangGraph needs to know HOW to combine those partial results.
# This function is used as a "reducer" — it merges two dicts, with 'b' winning on key conflicts.
def merge_findings(a: dict[str, str], b: dict[str, str]) -> dict[str, str]:
    # `b` overwrites `a` in case of conflicts.
    # This fragment can be modified to raise an error instead.
    return {**a, **b}

# Graph state for the parallel research pipeline
class ResearchState(TypedDict):
    topic: str                                             # The research question fed into the graph
    findings: Annotated[dict[str, str], merge_findings]   # Results from all sources, merged by merge_findings
    summary: str                                           # Final human-readable summary produced by the 'merge' node

In [ ]:
# --- Parallel data-fetching nodes ---
# These three nodes will be executed CONCURRENTLY because they all receive edges from START.
# Each simulates a slow network request using time.sleep(2).
# In a real system, these would call Wikipedia, a news API, and ArXiv respectively.

def wikipedia(state: ResearchState):
    print("[WIKIPEDIA] node is executing")
    time.sleep(2)  # Simulate a 2-second API call
    # Returns only the 'findings' field — the merge_findings reducer will merge this with other results
    return { "findings": { "wiki": f"[WIKIPEDIA] {state['topic']}" } }

def news(state: ResearchState):
    print("[LIVE NEWS] node is executing")
    time.sleep(2)  # Simulate a 2-second API call
    return { "findings": { "news": f"[LIVE NEWS] Sensational! {state['topic']}" } }

def arxiv(state: ResearchState):
    print("[ARXIV] node is executing")
    time.sleep(2)  # Simulate a 2-second API call
    return { "findings": { "arxiv": f"[ARXIV] The theoretical hyperchaotic entanglement of relativities related to \"{state['topic']}\"" } }

def merge(state: ResearchState):
    # This node runs AFTER all three parallel nodes finish (fan-in).
    # By the time it executes, state['findings'] already contains all merged results.
    bullets = "\n".join(f" - {finding}" for finding in state.get("findings", {}).values())
    return { "summary": f"Summary:\n{bullets}" }

In [ ]:
# Build the parallel research graph.
# KEY CONCEPT: Adding multiple edges FROM the same source node (here: START) creates a FAN-OUT —
# all target nodes will execute in parallel. LangGraph waits for ALL of them to finish
# before continuing to the next node (FAN-IN at 'merge').
graph_builder = StateGraph(ResearchState)
graph_builder.add_node("wikipedia", wikipedia)
graph_builder.add_node("news", news)
graph_builder.add_node("arxiv", arxiv)
graph_builder.add_node("merge", merge)

# Fan-out: all three research nodes start simultaneously
graph_builder.add_edge(START, "wikipedia")
graph_builder.add_edge(START, "news")
graph_builder.add_edge(START, "arxiv")

# Fan-in: 'merge' won't run until ALL three research nodes have completed
graph_builder.add_edge("wikipedia", "merge")
graph_builder.add_edge("news", "merge")
graph_builder.add_edge("arxiv", "merge")

graph_builder.add_edge("merge", END)

# Compile with a checkpointer — required to access state history later
graph = graph_builder.compile(checkpointer=checkpointer)

In [ ]:
# Visualize the fan-out/fan-in graph structure:
# START forks into 3 parallel branches, all converging at 'merge'
display_graph(graph, Path("/content/graph.png"))

In [ ]:
# Run the graph with a research topic.
# All three source nodes will start at the same time — total time ~2s instead of ~6s sequential.
thread1_config = { "configurable": { "thread_id": "thread_1" } }
thread1_result = graph.invoke(
    input={
        "topic": "Maximum snooker break"
    },
    config=thread1_config
)

In [ ]:
# Print the final summary produced by the 'merge' node
print(thread1_result['summary'])

In [ ]:
# Inspect the raw result dictionary — contains topic, findings (all sources merged), and summary
thread1_result

In [ ]:
# Explore the step-by-step state history.
# Notice that all three parallel nodes appear in the SAME step — confirming concurrent execution.
explore_state_history(graph, thread1_config)